In [1]:
import numpy as np
import os, glob
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Flatten
from keras.layers.convolutional import Conv2D
from keras.layers.convolutional import MaxPooling2D
from keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from keras.utils import to_categorical

Using TensorFlow backend.


In [2]:
model = Sequential()
model.add(Conv2D(32, kernel_size=(3, 3),
                 activation='relu',
                 input_shape=(24,24,3)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dense(3, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [3]:
img_ext = 'png'
circle = glob.glob('shapes/circle' + '/*' + img_ext)
rectangle = glob.glob('shapes/rectangle' + '/*' + img_ext)
triangle = glob.glob('shapes/triangle' + '/*' + img_ext)
img_names = circle + rectangle + triangle
img_names = [names.replace('\\', '/') for names in img_names]
np.random.shuffle(img_names)
train = img_names[:-12]
valid = img_names[-12:]
print("Found {} images.".format(len(img_names)))
print(" - train : {}".format(len(train)))
print(" - valid : {}".format(len(valid)))
labels = ['circle', 'rectangle', 'triangle']
batch_size = 3

Found 60 images.
 - train : 48
 - valid : 12


In [4]:
def train_generator():
    while True:
        for start in range(0, len(train), batch_size):
            x_batch = []
            y_batch = []
            end = min(start + batch_size, len(train))
            ids = train[start:end]
            for path in ids:
                img = load_img(path, color_mode='rgb', target_size=(24,24))
                img = img_to_array(img)
                label = path.split('/')[1]
                label = to_categorical(labels.index(label), len(labels))
                x_batch.append(img)
                y_batch.append(label)
            x_batch = np.array(x_batch, np.float32) / 255
            y_batch = np.array(y_batch, np.float32)
            yield (x_batch, y_batch)

In [5]:
def valid_generator():
    while True:
        for start in range(0, len(valid), batch_size):
            x_batch = []
            y_batch = []
            end = min(start + batch_size, len(valid))
            ids = valid[start:end]
            for path in ids:
                img = load_img(path, color_mode='rgb', target_size=(24,24))
                img = img_to_array(img)
                label = path.split('/')[1]
                label = to_categorical(labels.index(label), len(labels))
                x_batch.append(img)
                y_batch.append(label)
            x_batch = np.array(x_batch, np.float32) / 255
            y_batch = np.array(y_batch, np.float32)
            yield (x_batch, y_batch)

In [7]:
model.fit_generator(
        train_generator(),
        steps_per_epoch=len(train),
        epochs=5,
        validation_data=valid_generator(),
        validation_steps=len(valid)
)

Epoch 1/5
48/48 [==============================] - 2s 40ms/step - loss: 2.8570e-04 - acc: 1.0000 - val_loss: 0.0038 - val_acc: 1.0000
Epoch 2/5
48/48 [==============================] - 2s 37ms/step - loss: 1.5016e-04 - acc: 1.0000 - val_loss: 0.0033 - val_acc: 1.0000
Epoch 3/5
48/48 [==============================] - 2s 36ms/step - loss: 1.0102e-04 - acc: 1.0000 - val_loss: 0.0031 - val_acc: 1.0000
Epoch 4/5
48/48 [==============================] - 2s 36ms/step - loss: 7.3714e-05 - acc: 1.0000 - val_loss: 0.0029 - val_acc: 1.0000
Epoch 5/5
48/48 [==============================] - 2s 37ms/step - loss: 5.6825e-05 - acc: 1.0000 - val_loss: 0.0028 - val_acc: 1.0000


In [12]:
def predict(img_path, model):
    img = load_img(img_path)
    img = img_to_array(img)/255
    img = np.expand_dims(img, axis=0)
    return model.predict(img)

In [8]:
test_path = ['shapes/circle/circle015.png', 'shapes/rectangle/rectangle015.png', 'shapes/triangle/triangle015.png']

In [13]:
a = [predict(test, model) for test in test_path]

In [15]:
for result in a:
    print('Prediction : {}'.format(labels[np.argmax(result)]))

Prediction : circle
Prediction : rectangle
Prediction : triangle
